In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, random_split

In [ ]:
class CustomDataset(Dataset):
    
    # 데이터 전처리
    def __init__(self, window_size, folder_dir):
        self.data_list = []
        self.stride = 2
        self.window_size = window_size
        self.folder_path = folder_dir

        # 주어진 폴더에서 파일 읽어오기
        for filename in os.listdir(self.folder_path):
            data = pd.read_csv(os.path.join(self.folder_path, filename), header=None, encoding='utf-8').to_numpy()

            features = data[:,0:78].astype(np.float32)
            features = torch.FloatTensor(features)
           
            labels = data[:,78].astype(np.float32)
            labels = torch.FloatTensor(labels)
            labels = labels.reshape(len(labels),-1)
                     
            # 윈도우 크기에 따라 데이터 전처리         
            for i in range(0, len(features)-self.window_size+1, self.stride):
                features_subset = features[i:i+self.window_size]
                labels_subset = labels[i]
                self.data_list.append([features_subset,labels_subset])


    # 데이터 길이 반환
    def __len__(self):
        return len(self.data_list)


    # 데이터 셋을 전처리 후 반환
    def __getitem__(self, idx):
        x, y = self.data_list[idx]
        x = torch.FloatTensor(x)
        y = torch.FloatTensor(y)
        return x, y

In [ ]:
# 텐서 변경
dataset = CustomDataset(window_size=30, folder_dir='../FGCS_tmp/total')

print(len(dataset))
print(dataset[0][0].shape)
print(type(dataset))

In [ ]:
# train, test, validation 데이터셋 분리
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size 
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

batch_size = 64
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
for train_x, train_y in train_loader:
    print(train_x.shape, train_y.shape)
    break

In [ ]:
for test_x, test_y in test_loader:
    print(test_x.shape, test_y.shape)
    break

In [ ]:
import random
from cnn_lstm import CNN_LSTM


device = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(0)
torch.manual_seed(0)

if device == 'cuda':
    torch.cuda.manual_seed_all(0)
    print('cuda ready')

CNN_LSTM_model = CNN_LSTM(input_size=78,
                          output_size=128,  
                          hidden_size=64,
                          num_classes=30).to(device)

In [ ]:
from torchinfo import summary
summary(CNN_LSTM_model, (128,30,78))

In [ ]:
# 모델 학습을 위한 변수 지정 
num_epochs = 200
learning_rate = 0.0001  

model = CNN_LSTM(input_size = train_x.shape[2],
                output_size = train_x.shape[0],   
                 hidden_size = 32).to(device)

# optimizer 설정
optimizer = torch.optim.Adam(CNN_LSTM_model.parameters(), lr=learning_rate)
# loss 함수
criterion = nn.CrossEntropyLoss().to(device)

In [ ]:
from tqdm import tqdm
from model.Earlystopping import EarlyStopping

def train_model(model,train_df,val_df,criterion,optimizer,num_epochs):   
    train_loss = []
    valid_loss = []
    train_acc = []
    valid_acc = []

    early_stopping = EarlyStopping(patience = 20, verbose = True)
    
    for epoch in tqdm(range(num_epochs)):
        model.train()
        train_running_loss = 0.0
        correct = 0

        for _, (x, y) in enumerate(train_df):
            y = y.type(torch.LongTensor)
            x_train = x.to(device)
            y_train = y.to(device)
            
            # H(x) 계산
            output = model(x_train)

            # cost 계산
            loss = criterion(output, y_train.squeeze(dim=-1))   
            train_running_loss += loss.item()                 
            
            # cost로 H(x) 개선
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # 정확도 계산
            _, indices = torch.max(output.data, dim=1, keepdim=True)
            correct += (indices == y_train).sum().item()
            
        acc = 100. * correct/len(train_df.dataset)
        train_acc.append(acc)
        epoch_loss = train_running_loss/len(train_df.dataset)
        train_loss.append(epoch_loss)

        print('Epoch:', '%04d' % (epoch+1), 'train loss : {:.6f}, Accuracy: {:.2f}%'.format(epoch_loss, acc))
        
        model.eval()
        val_running_loss=0.0
        val_correct =0
        with torch.no_grad():
            for batch_idx, (x,y) in enumerate(val_df): 
                y = y.type(torch.LongTensor)
                x_val = x.to(device)
                y_val = y.to(device)

                # H(x) 계산
                output = model(x_val)
                # cost 계산
                loss = criterion(output, y_val.squeeze(dim=-1))   
                val_running_loss += loss.item() 

                # 정확하게 분류한 샘플 수 계산
                values, indices = torch.max(output.data, dim=1, keepdim=True)        
                val_correct += (indices == y_val).sum().item()

            epoch_valid_loss = val_running_loss/len(val_df.dataset)
            valid_loss.append(epoch_valid_loss)

            val_accuracy = 100 * val_correct / len(val_df.dataset)
            valid_acc.append(val_accuracy)  
            print('Validation set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(epoch_valid_loss, val_accuracy))

            early_stopping(epoch_valid_loss, model)
        
            if early_stopping.early_stop:
                print("Early stopping")
                break

    return train_loss, valid_loss, train_acc, valid_acc 

In [ ]:
train_losses, valid_losses, train_acc, valid_acc = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=num_epochs)

In [ ]:
model.eval()

test_loss = 0.0
test_correct = 0

with torch.no_grad():
    for x_test, y_test in test_loader : 
        y_test = y_test.type(torch.LongTensor)
        x_test, y_test = x_test.to(device), y_test.to(device)
        test_output = model(x_test)

        # cost 계산
        loss = criterion(test_output, y_test.squeeze(dim=-1))   
        test_loss += loss.item() 

        pred = test_output.argmax(dim=1, keepdim = True)
        test_correct += pred.eq(y_test.view_as(pred)).sum().item()

test_loss /= len(test_loader.dataset)

print('Test set: Average Loss: {:.4f}, Accuracy: {}/{} ( {:.0f}%)\n'.format(
test_loss, test_correct, len(test_loader.dataset), 100 * test_correct / len(test_loader.dataset)))

In [ ]:
import matplotlib.pyplot as plt

def plot_loss(train_loss, valid_loss):
    plt.plot(train_loss, label='Training Loss')
    plt.plot(valid_loss, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.show()

def plot_accuracy(train_acc, valid_acc):
    plt.plot(train_acc, label='Training Accuracy')
    plt.plot(valid_acc, label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.show()

plot_loss(train_losses, valid_losses)
plot_accuracy(train_acc, valid_acc)